In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

# Loading the data
train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

# Handle Missing Values

train['sleep_hours'].fillna(train['sleep_hours'].median(), inplace=True)
train['previous_day_mood'].fillna(train['previous_day_mood'].mode()[0], inplace=True)
train['face_emotion_hint'].fillna("unknown", inplace=True)

test['sleep_hours'].fillna(train['sleep_hours'].median(), inplace=True)
test['previous_day_mood'].fillna(train['previous_day_mood'].mode()[0], inplace=True)
test['face_emotion_hint'].fillna("unknown", inplace=True)

# Defining Features

text_col = 'journal_text'

categorical_cols = [
    'ambience_type', 'time_of_day',
    'previous_day_mood', 'face_emotion_hint',
    'reflection_quality'
]

numerical_cols = [
    'duration_min', 'sleep_hours',
    'energy_level', 'stress_level'
]

# Targets
y_state = train['emotional_state']
y_intensity = train['intensity']

X = train.drop(columns=['id', 'emotional_state', 'intensity'])
X_test = test.drop(columns=['id'])

# Preprocessing

preprocessor = ColumnTransformer(
    transformers=[
        ('text', TfidfVectorizer(max_features=5000), text_col),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols),
        ('num', 'passthrough', numerical_cols)
    ]
)

# Models

# Emotional State Model
model_state = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])

# Intensity Model
model_intensity = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(max_iter=1000))
])

# Train

model_state.fit(X, y_state)
model_intensity.fit(X, y_intensity)
import joblib
import os

os.makedirs("../outputs", exist_ok=True)

joblib.dump(model_state, "../outputs/model_state.pkl")
joblib.dump(model_intensity, "../outputs/model_intensity.pkl")

print("Models saved successfully!")

# 6.Predict

pred_state = model_state.predict(X_test)
pred_intensity = model_intensity.predict(X_test)

# 7.Output

output = pd.DataFrame({
    'id': test['id'],
    'predicted_state': pred_state,
    'predicted_intensity': pred_intensity
})

output.to_csv("predictions.csv", index=False)

print("Done!")

# Model Metric Evaluation

from sklearn.metrics import accuracy_score

# Spilting the data (for evaluation)

X_train, X_val, y_train, y_val = train_test_split(X, y_state, test_size=0.2, random_state=42)

# 1.Text only model

text_vectorizer = TfidfVectorizer(max_features=5000)
X_train_text = text_vectorizer.fit_transform(X_train[text_col])
X_val_text = text_vectorizer.transform(X_val[text_col])
text_model = LogisticRegression(max_iter=1000)
text_model.fit(X_train_text, y_train)
pred_text = text_model.predict(X_val_text)
acc_text = accuracy_score(y_val, pred_text)
print("Text-only Accuracy:", acc_text)

# 2.Meta data only model

meta_cols = categorical_cols + numerical_cols
# simple encoding for categorical
X_train_meta = pd.get_dummies(X_train[meta_cols])
X_val_meta = pd.get_dummies(X_val[meta_cols]) 
# align columns (important)
X_train_meta, X_val_meta = X_train_meta.align(X_val_meta, join='left', axis=1, fill_value=0)
meta_model = LogisticRegression(max_iter=1000)
meta_model.fit(X_train_meta, y_train)
pred_meta = meta_model.predict(X_val_meta)
acc_meta = accuracy_score(y_val, pred_meta)
print("Metadata-only Accuracy:", acc_meta)

# 3.Full model (already exists)

# reusing the pipeline

full_model = model_state
full_model.fit(X_train, y_train)
pred_full = full_model.predict(X_val)
acc_full = accuracy_score(y_val, pred_full)
print("Full Model Accuracy:", acc_full)


# 4.Final Model Comparsion

print("\n FINAL COMPARISON")
print(f"Text-only      : {acc_text:.4f}")
print(f"Metadata-only  : {acc_meta:.4f}")
print(f"Text + Metadata: {acc_full:.4f}")
print("\n Insight:")
print("Text = primary signal")
print("Metadata = supporting context")


# ERROR ANALYSIS

print("\n🔍 ERROR ANALYSIS STARTED\n")

# Merge validation predictions with actual labels
val_df = X_val.copy()
val_df['actual_state'] = y_val.values
val_df['predicted_state'] = pred_full

# Find wrong predictions
errors = val_df[val_df['actual_state'] != val_df['predicted_state']]

print(f"Total validation samples: {len(val_df)}")
print(f"Total errors: {len(errors)}\n")

# Show 6 real failure cases
print("🔹 REAL FAILURE CASES (from dataset)\n")

sample_errors = errors[['journal_text', 'actual_state', 'predicted_state']].head(6)

for i, row in sample_errors.iterrows():
    print("------ CASE ------")
    print("Text:", row['journal_text'])
    print("Actual:", row['actual_state'])
    print("Predicted:", row['predicted_state'])
    
    # Simple reasoning
    text = str(row['journal_text']).lower()
    
    if len(text.split()) <= 2:
        reason = "Very short text → insufficient signal"
    elif "but" in text or "and" in text:
        reason = "Conflicting emotions in sentence"
    elif any(word in text for word in ["ok", "fine", "nothing"]):
        reason = "Ambiguous / neutral wording"
    else:
        reason = "Subtle emotional context missed"
    
    print("Reason:", reason)
    print("Suggestion: Improve context understanding / use better embeddings\n")


# MANUAL EDGE CASES

print("\n MANUAL EDGE CASES\n")

edge_cases = [
    "ok",
    "fine",
    "I am tired but excited",
    "nothing much happened",
    "very stressed but trying to stay calm"
]

for text in edge_cases:
    print("------ EDGE CASE ------")
    print("Text:", text)
    
    if len(text.split()) <= 2:
        print("Issue: Too short → model lacks signal")
    elif "but" in text:
        print("Issue: Conflicting emotions")
    else:
        print("Issue: Ambiguous meaning")
    
    print("Suggestion: Add fallback rules or context tracking\n")

print("ERROR ANALYSIS COMPLETED")




/tmp/ipykernel_13831/2393016326.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train['sleep_hours'].fillna(train['sleep_hours'].median(), inplace=True)
/tmp/ipykernel_13831/2393016326.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method

Models saved successfully!
Done!
Text-only Accuracy: 0.6458333333333334
Metadata-only Accuracy: 0.1625
Full Model Accuracy: 0.575

 FINAL COMPARISON
Text-only      : 0.6458
Metadata-only  : 0.1625
Text + Metadata: 0.5750

 Insight:
Text = primary signal
Metadata = supporting context

🔍 ERROR ANALYSIS STARTED

Total validation samples: 240
Total errors: 102

🔹 REAL FAILURE CASES (from dataset)

------ CASE ------
Text: lowkey felt pretty even, but this was better than yesterday. ...
Actual: neutral
Predicted: mixed
Reason: Conflicting emotions in sentence
Suggestion: Improve context understanding / use better embeddings

------ CASE ------
Text: lowkey felt locked in for a bit, but mountain visuals made it easier to pause.
Actual: focused
Predicted: restless
Reason: Conflicting emotions in sentence
Suggestion: Improve context understanding / use better embeddings

------ CASE ------
Text: honestly i felt lighter than before. i couldn't tell if it was helping at first.
Actual: calm
Predi